# End-to-End Customer Intelligence System

This project builds a machine learning-based system to predict churn and recommend business actions.

In [33]:
import pandas as pd
import numpy as np

In [34]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [35]:
df['Churn'] = df['Churn'].map({'Yes':1, 'No':0})
df.drop('customerID', axis=1, inplace=True)

In [36]:
df = pd.get_dummies(df, drop_first=True)

In [37]:
from sklearn.model_selection import train_test_split

X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [38]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [39]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

In [40]:
def segment(prob):
    if prob > 0.7:
        return "High Risk"
    elif prob > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

In [41]:
df['Churn_Probability'] = model.predict_proba(X)[:,1]
df['Risk_Segment'] = df['Churn_Probability'].apply(segment)

In [42]:
df[['Churn_Probability','Risk_Segment']].head()

,Churn_Probability,Risk_Segment
0,0.27,Low Risk
1,0.00,Low Risk
2,0.67,Medium Risk
3,0.03,Low Risk
4,0.83,High Risk


In [43]:
# @title
def action(seg):
    if seg == "High Risk":
        return "Give Discount"
    elif seg == "Medium Risk":
        return "Send Promotion"
    else:
        return "Upsell"

df['Recommended_Action'] = df['Risk_Segment'].apply(action)

In [44]:
df[['Risk_Segment','Recommended_Action']].head()

,Risk_Segment,Recommended_Action
0,Low Risk,Upsell
1,Low Risk,Upsell
2,Medium Risk,Send Promotion
3,Low Risk,Upsell
4,High Risk,Give Discount


## Business Insights

- High risk customers should receive targeted discounts  
- Medium risk customers can be engaged with promotions  
- Low risk customers are suitable for upselling strategies  

This system enables data-driven decision making for customer retention.

In [46]:
df = df.drop(columns=['Risk_Segment', 'Recommended_Action'], errors='ignore')

In [47]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [48]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [49]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [50]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7977288857345636


In [51]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.92      0.87      1036
           1       0.67      0.47      0.55       373

    accuracy                           0.80      1409
   macro avg       0.75      0.69      0.71      1409
weighted avg       0.79      0.80      0.79      1409



In [52]:
y_prob = model.predict_proba(X_test)[:,1]

In [53]:
y_prob[:10]

array([0.85      , 0.04      , 0.        , 0.875     , 0.05      ,
       0.13      , 0.01      , 0.        , 0.06      , 0.01666667])

In [54]:
X_test = X_test.copy()
X_test['Churn_Probability'] = y_prob

In [55]:
X_test[['Churn_Probability']].head()

,Churn_Probability
185,0.850
2715,0.040
3825,0.000
1807,0.875
132,0.050


In [56]:
def segment(prob):
    if prob > 0.7:
        return "High Risk"
    elif prob > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

In [57]:
X_test['Risk_Segment'] = X_test['Churn_Probability'].apply(segment)

In [58]:
X_test[['Churn_Probability','Risk_Segment']].head(10)

,Churn_Probability,Risk_Segment
185,0.850000,High Risk
2715,0.040000,Low Risk
3825,0.000000,Low Risk
1807,0.875000,High Risk
132,0.050000,Low Risk
1263,0.130000,Low Risk
3732,0.010000,Low Risk
1672,0.000000,Low Risk
811,0.060000,Low Risk
2526,0.016667,Low Risk


In [59]:
X_test['Risk_Segment'].value_counts()

,count
Risk_Segment,
Low Risk,1132
High Risk,232
Medium Risk,45


In [60]:
def action(seg):
    if seg == "High Risk":
        return "Offer Discount"
    elif seg == "Medium Risk":
        return "Send Promotion"
    else:
        return "Upsell Opportunity"

In [61]:
X_test['Recommended_Action'] = X_test['Risk_Segment'].apply(action)

In [62]:
X_test[['Churn_Probability','Risk_Segment','Recommended_Action']].head(10)

,Churn_Probability,Risk_Segment,Recommended_Action
185,0.850000,High Risk,Offer Discount
2715,0.040000,Low Risk,Upsell Opportunity
3825,0.000000,Low Risk,Upsell Opportunity
1807,0.875000,High Risk,Offer Discount
132,0.050000,Low Risk,Upsell Opportunity
1263,0.130000,Low Risk,Upsell Opportunity
3732,0.010000,Low Risk,Upsell Opportunity
1672,0.000000,Low Risk,Upsell Opportunity
811,0.060000,Low Risk,Upsell Opportunity
2526,0.016667,Low Risk,Upsell Opportunity


## Business Conclusion

This system enables companies to identify high-risk customers and take proactive actions such as offering discounts or targeted promotions.

By combining machine learning with business rules, organizations can improve customer retention and optimize revenue strategies.